# Resume a voice investigation without starting twice

A voice connection is temporary; an investigation is application work with its
own identity. This offline recipe makes reconnect a **read-only attach** to the
same investigation, then resumes a single leased poller.

**You will learn:** keep a stable intent key and investigation ID, persist
per-job event progress, fence stale polling owners, and expose unknown delivery
rather than claiming an interrupted utterance was or was not heard.

Draft contribution by Sophia Cole / ColeWorld; MIT licensed (see LICENSE). Not accepted or endorsed by OpenAI. All data,
the investigator, and the transport are fictional. There are no model/API calls,
keys, sockets, audio devices, paid services, or external writes. Local SQLite
writes go to a temporary directory. Python 3.11+ is sufficient.

## 1. Separate work identity from connection identity

Use a new intent key only when the user deliberately starts new work. Persist it
before submitting the start. Retrying a lost response uses that same key and
prompt. An investigation ID belongs to the job, while a polling owner belongs
to one connection/process.

`Scope` below represents a server-authenticated tenant. It is a test seam, not
an authentication implementation. Production callers must not choose a tenant
by supplying an arbitrary string.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
from continuity import Event, MockTransport, OwnershipLost, Poller, Scope, Store, fictional_events

scratch = TemporaryDirectory(prefix="fictional-investigation-")
database = Path(scratch.name) / "continuity.sqlite3"
store = Store(database)
scope = Scope("fictional-team")
clock = [1000.0]  # Controlled authoritative time for deterministic examples.
intent_key = "persisted-client-intent-17"
prompt = "Investigate the fictional search latency increase."

job = store.start_or_attach(scope, intent_key, prompt)
investigation_id = job.investigation_id  # Persist in application/session state.
retry = store.start_or_attach(scope, intent_key, prompt)
assert retry.investigation_id == investigation_id
print("A repeated start reused the same investigation ID.")

## 2. Make reconnect an attach, never an implicit start

Reading the existing job must not dispatch the investigator or create a new job
if the ID is missing. A different prompt with an old intent key is a conflict;
a genuinely new investigation requires a new intent key.

The mock database insert is the entire start operation here. For a real remote
provider, application idempotency does **not** automatically prevent duplicate
provider calls. Use a verified idempotent provider operation or a transactional
dispatch/reconciliation design; preserve an unknown start outcome when a lost
response cannot be reconciled.

In [ ]:
changes_before_attach = store.db.total_changes
attached = store.attach(scope, investigation_id)
assert attached.investigation_id == investigation_id
assert store.db.total_changes == changes_before_attach
print("Attach performed no writes and started no new work.")

## 3. Persist a contiguous, immutable event history

The fictional read-only investigator has produced two updates. Sequence numbers
are per investigation, start at 1, and are independent of the voice connection.
An identical replay is deduplicated. A gap, changed replay, or new event after
completion is rejected rather than silently advancing a cursor.

In [ ]:
source_events = fictional_events()
store.ingest(scope, investigation_id, source_events[:2])
store.ingest(scope, investigation_id, source_events[:2])  # Harmless replay.
assert store.attach(scope, investigation_id).status == "running"
print("Two source events persisted; their replay added no duplicates.")

## 4. Give one poller a fenced lease

SQLite admits one unexpired owner for the job. Every reservation, acknowledgment,
renewal, and disconnect checks both the owner and its generation (fence).

Reservation and cursor advancement happen in the same transaction **before**
emission. The cursor means "reserved for an attempted update," not "spoken".
The sink's receipt means only that this fictional application accepted it.

In [ ]:
first_lease = store.claim(scope, investigation_id, "connection-a", clock[0], ttl=10)
try:
    store.claim(scope, investigation_id, "connection-b", clock[0], ttl=10)
except OwnershipLost:
    print("A second live poller was rejected.")
else:
    raise AssertionError("Only one current polling owner should be accepted")

first_poller = Poller(store, scope, first_lease, lambda: clock[0])
first_sink = MockTransport()
assert first_poller.once(first_sink) == "acknowledged"
assert store.attach(scope, investigation_id).cursor == 1
print("Update 1 has an application receipt, not proof of audible delivery.")

## 5. Crash after emission, before storing the acknowledgment

Both of these interruptions can leave the same durable state:

- Reserve an update, then crash before sending it.
- Send an update, then crash before storing its receipt.

There is no database transaction spanning an audio speaker. The safe statement
is **delivery unknown**, not "it failed, so repeat it" or "it was definitely
spoken." This policy can omit an update that was never emitted; that tradeoff
is explicit.

In [ ]:
interrupted_sink = MockTransport("crash_after_emit")
try:
    first_poller.once(interrupted_sink)
except RuntimeError as error:
    assert "Simulated process crash" in str(error)
else:
    raise AssertionError("The test transport should simulate a crash")

assert interrupted_sink.accepted == [f"{investigation_id}:2"]
assert store.attach(scope, investigation_id).cursor == 2
status = store.resume_status(scope, investigation_id)
assert status["delivery_outcome"] == "unknown"
assert status["uncertain_sequences"] == [2]
print(status["message"])
store.close()  # Simulate losing the original application's connection.

## 6. The job can finish while the voice connection is gone

A worker writes the final result to the same local job. This demonstrates
independent lifetimes; it does not dispatch a real background worker or model.

In [ ]:
worker_store = Store(database)
worker_store.ingest(scope, investigation_id, source_events)  # Replay + final event.
assert worker_store.attach(scope, investigation_id).status == "completed"
worker_store.close()
print("The existing investigation completed while the voice session was offline.")

## 7. Reconnect, reconcile uncertainty, then continue

The new connection attaches using the persisted investigation ID. It reads the
job and unresolved delivery state without starting work. After the old lease
expires, a new owner claims a higher fence and marks unresolved attempts unknown.

No automatic replay of update 2 occurs. The next reservation is update 3. A real
UI should show the unknown-delivery notice and can let the user explicitly ask
for a fresh summary. That user-requested summary is separate from silently
replaying an ambiguous utterance.

In [ ]:
store = Store(database)
reconnected = store.attach(scope, investigation_id)
assert reconnected.investigation_id == investigation_id
assert reconnected.status == "completed"
assert reconnected.cursor == 2
assert store.resume_status(scope, investigation_id)["uncertain_sequences"] == [2]

clock[0] = 1011.0  # The previous lease expired at 1010.
next_lease = store.claim(scope, investigation_id, "connection-b", clock[0], ttl=10)
assert next_lease.fence > first_lease.fence
try:
    store.acknowledge(scope, first_lease, 2, clock[0])
except OwnershipLost:
    print("A stale owner could not commit a late acknowledgment.")
else:
    raise AssertionError("The stale owner must be fenced")

next_sink = MockTransport()
next_poller = Poller(store, scope, next_lease, lambda: clock[0])
assert next_poller.once(next_sink) == "acknowledged"
assert next_poller.once(next_sink) == "idle"
assert next_sink.accepted == [f"{investigation_id}:3"]
assert [(d.event.sequence, d.state) for d in store.deliveries(scope, investigation_id)] == [
    (1, "acknowledged"), (2, "unknown"), (3, "acknowledged")
]
print("Same investigation; update 3 continued; update 2 remains explicitly unknown.")

## What this does and does not guarantee

Within this local SQLite example, duplicate start requests share one job,
accepted poller operations are fenced, replayed source events are deduplicated,
and reservations/cursors survive reopening storage. The test suite additionally
opens the database from a separately launched Python process.

It does **not** guarantee exactly-once model execution or spoken delivery. A
stale process may already hold a prepared update and can send after losing its
lease. Checking the lease immediately before emission narrows, but cannot remove,
that race. A real sink needs its own fencing/deduplication contract. Even an
application receipt cannot prove a person heard audio.

This recipe does not implement Realtime handshakes, WebRTC, authentication,
remote-job APIs, production queues, distributed clocks, database failover, or
approval flows. An adapter must use current documented provider contracts,
including genuine replay and acknowledgment semantics. No adapter or invented
API method is supplied here.

Run the offline tests with `python -m unittest -v test_continuity`. Run
`python validate_notebook.py --require-schema` after installing `requirements.txt`
to validate schema and sequential cells. That check is **not** full Jupyter
kernel execution. See the README for failure windows and production boundaries.

In [ ]:
final_status = store.resume_status(scope, investigation_id)
assert final_status["job_status"] == "completed"
assert final_status["delivery_outcome"] == "unknown"
store.close()
scratch.cleanup()
print("Temporary database cleaned up. No external calls or writes were made.")